In [ ]:
# ── LOAD CHECKPOINT ────────────────────────────────────────────────
# Point this at your saved .pth file
CHECKPOINT_PATH = f"{DATA_PATH}/checkpoints/YOUR_CHECKPOINT.pth"  # <-- update filename

import torch, torch.nn as nn
import timm

class MemorabilityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('resnet50', pretrained=False, num_classes=0)
        self.regressor = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.regressor(self.backbone(x)).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MemorabilityModel().to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
print(f"Checkpoint loaded on: {device}")

# ── SHARED TRANSFORM (same as training) ────────────────────────────
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── SHARED PREDICT FUNCTION ─────────────────────────────────────────
from PIL import Image
import numpy as np

def predict_score(pil_img):
    """Return memorability score (0-1) for a single PIL image."""
    model.eval()
    with torch.no_grad():
        t = transform(pil_img).unsqueeze(0).to(device)
        return model(t).item()

In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        output = self.model(input_tensor)  # forward pass
        self.model.zero_grad()
        output.backward()                  # backward pass
        
        # Pool gradients over spatial dims
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # normalize

# --- Usage ---
target_layer = model.backbone.layer4[-1]  # last ResNet block
gradcam = GradCAM(model, target_layer)

def show_heatmap(image_path, model, transform, device):
    pil_img = Image.open(image_path).convert('RGB')
    tensor = transform(pil_img).unsqueeze(0).to(device).requires_grad_(True)
    
    cam = gradcam.generate(tensor)
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(pil_img); axes[0].set_title("Original"); axes[0].axis('off')
    axes[1].imshow(cam, cmap='jet'); axes[1].set_title("Grad-CAM"); axes[1].axis('off')
    # Overlay
    axes[2].imshow(pil_img)
    axes[2].imshow(cam, cmap='jet', alpha=0.45)
    axes[2].set_title("Overlay"); axes[2].axis('off')
    plt.tight_layout(); plt.show()